# Module 4: Matplotlib Basics

**Course:** Matplotlib Training — 40-Day Bootcamp  
**Dataset:** CHIRPS Rainfall (Ethiopia, 1981–2022)  
**Objective:** Master the matplotlib object-oriented API — figures, axes, plotting, and saving.

---
## Learning Objectives

- Distinguish between pyplot (state-based) and OO (explicit) interfaces
- Understand `Figure` and `Axes` objects
- Follow the basic plotting workflow: `fig, ax = plt.subplots()`, `ax.plot()`, `plt.show()`
- Customise plot elements: title, labels, colours, line styles, markers
- Save figures with `savefig()` — `dpi`, formats, `bbox_inches`
- Choose figure resolution and formats for different use cases
- Plot a CHIRPS time series at a single grid point

---
## 1. pyplot vs Object-Oriented Interface

Matplotlib offers two interfaces:

### pyplot (state-based)
```python
import matplotlib.pyplot as plt
plt.plot(x, y)
plt.title('Rainfall')
plt.show()
```
- Implicit: `plt` keeps track of the "current" figure and axes
- Convenient for quick scripts and interactive use
- **Not recommended** for complex figures, subplots, or reproducibility

### Object-Oriented (explicit)
```python
fig, ax = plt.subplots()
ax.plot(x, y)
ax.set_title('Rainfall')
plt.show()
```
- Explicit: you control `fig` (the window) and `ax` (the plot area)
- **Preferred** for this course — scales to any complexity
- Easier to reuse in functions, scripts, and libraries

> **Rule of thumb:** Always use the OO interface. You will thank yourself when you need subplots.

---
## 2. The Figure and Axes Objects

### Figure (`fig`)
- The **top-level container** that holds everything
- Think of it as the **canvas** or **window**
- Attributes: `figsize`, `dpi`, `facecolor`
- Methods: `savefig()`, `suptitle()`, `colorbar()`

### Axes (`ax`)
- The **actual plotting area** — this is where data is drawn
- Think of it as a **panel** inside the figure
- A figure can have one or many axes (subplots)
- Attributes: x-axis, y-axis, title, labels
- Methods: `plot()`, `scatter()`, `pcolormesh()`, `hist()`, `set_title()`, `set_xlabel()`

In [ ]:
# Anatomy of a Figure
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(8, 5))
print(f'Figure type: {type(fig)}')
print(f'Empty figure — nothing plotted yet')
plt.close(fig)

In [ ]:
# Creating a Figure with one Axes
fig, ax = plt.subplots(figsize=(8, 5))
print(f'Figure type: {type(fig)}')
print(f'Axes type:   {type(ax)}')
plt.close(fig)

In [ ]:
# Creating a Figure with multiple subplots (grid of axes)
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
print(f'Axes shape: {axes.shape}')
print(f'Top-left:   {type(axes[0, 0])}')
plt.close(fig)

---
## 3. Basic Plotting Workflow

The standard workflow is:

```python
# 1. Create figure and axes
fig, ax = plt.subplots(figsize=(width, height))

# 2. Plot data onto the axes
ax.plot(x, y, <style>)

# 3. Customise labels, title, legend
ax.set_xlabel('...')
ax.set_ylabel('...')
ax.set_title('...')

# 4. (Optional) Add extras — grid, legend, colourbar
ax.grid(True)
ax.legend()

# 5. Adjust layout and display
fig.tight_layout()
plt.show()
```

In [ ]:
# Load CHIRPS data for plotting
import xarray as xr
import numpy as np

ds = xr.open_dataset('../chirps_1981_2022.nc')

# Extract time series at a point near Addis Ababa
lat_addis, lon_addis = 9.0, 38.7
ts = ds.precip.sel(latitude=lat_addis, longitude=lon_addis, method='nearest')
years = ts.groupby('time.year').mean()

print(f'Location: lat={float(ts.latitude.values):.3f}, lon={float(ts.longitude.values):.3f}')
print(f'Annual mean: {float(years.mean().values):.1f} mm/month')

In [ ]:
# Step-by-step: annual rainfall time series
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(years.year.values, years.values,
        color='steelblue', linewidth=2, marker='o', markersize=4)

ax.set_title('Annual Rainfall Near Addis Ababa\nCHIRPS 1981–2022',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Mean Monthly Precipitation (mm/month)', fontsize=12)
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()

---
## 4. Customising Plot Elements

### Line styles and colours

| Argument | Options | Example |
|----------|---------|--------|
| `color` | Named, hex, RGB tuple | `'steelblue'`, `'#FF5733'` |
| `linestyle` | `'-'`, `'--'`, `':'`, `'-.'`, `'None'` | `ax.plot(x, y, linestyle='--')` |
| `linewidth` | Float (default 1.5) | `ax.plot(x, y, linewidth=2)` |
| `marker` | `'o'`, `'s'`, `'^'`, `'D'`, `'.'` | `ax.plot(x, y, marker='o')` |
| `markersize` | Float | `ax.plot(x, y, markersize=5)` |
| `alpha` | 0 (transparent) to 1 (opaque) | `ax.plot(x, y, alpha=0.7)` |
| `label` | String (for legend) | `ax.plot(x, y, label='CHIRPS')` |

### Axes methods

| Method | Purpose |
|--------|---------|
| `ax.set_xlim(left, right)` | Set x-axis range |
| `ax.set_ylim(bottom, top)` | Set y-axis range |
| `ax.set_xticks([...])` | Set tick locations |
| `ax.set_xticklabels([...])` | Set tick labels |
| `ax.tick_params(axis='x', rotation=45)` | Rotate tick labels |
| `ax.legend(loc='best')` | Show legend |
| `ax.grid(True, alpha=0.3)` | Add gridlines |
| `ax.axhline(y=value, color='r', ls='--')` | Add horizontal line |
| `ax.axvline(x=value, color='r', ls='--')` | Add vertical line |
| `ax.annotate('text', xy=(x,y))` | Add annotation |

In [ ]:
# Full customisation example
fig, ax = plt.subplots(figsize=(10, 5))

# Plot with multiple style options
ax.plot(years.year.values, years.values,
        color='#2C3E50', linestyle='-', linewidth=2,
        marker='s', markersize=5, markerfacecolor='#E74C3C',
        label='Mean annual rainfall')

# Long-term mean reference line
ax.axhline(y=float(years.mean().values), color='gray', linestyle='--',
           linewidth=1, alpha=0.7, label=f'Mean = {float(years.mean().values):.1f} mm/month')

# Customise
ax.set_title('Annual Rainfall Near Addis Ababa — Customised', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Precipitation (mm/month)', fontsize=12)
ax.set_xlim(1980, 2023)
ax.set_xticks(range(1980, 2024, 5))
ax.grid(alpha=0.3, which='both')
ax.legend(fontsize=11, loc='lower right')

fig.tight_layout()
plt.show()

---
## 5. Saving Figures

Use `fig.savefig()` — not `plt.savefig()` — for the OO interface.

### Key parameters:

| Parameter | Purpose | Example |
|-----------|---------|--------|
| `fname` | Filename (extension determines format) | `'rainfall.png'` |
| `dpi` | Resolution in dots per inch | `dpi=150` (screen), `dpi=300` (print) |
| `bbox_inches='tight'` | Remove whitespace around figure | `bbox_inches='tight'` |
| `facecolor` | Background colour | `facecolor='white'` |
| `transparent` | Transparent background | `transparent=True` |

### Supported formats:

| Format | Extension | Use Case |
|--------|-----------|----------|
| PNG | `.png` | Web, screen, presentations |
| PDF | `.pdf` | Publications, LaTeX, vector graphics |
| SVG | `.svg` | Vector, web, editable in illustrator |
| EPS | `.eps` | LaTeX (legacy) |
| JPG | `.jpg` / `.jpeg` | Photographs (lossy, not ideal for charts) |
| TIFF | `.tiff` | Scientific publishing (some journals) |

### DPI recommendations:
- **Web / screen:** 100–150 dpi
- **Print / publication:** 300 dpi
- **Poster:** 600+ dpi

In [ ]:
# Create a figure and save in multiple formats
import os

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(years.year.values, years.values, color='steelblue', linewidth=1.5)
ax.set_title('Annual Rainfall — Addis Ababa Region', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Precipitation (mm/month)')
ax.grid(alpha=0.3)

fig.tight_layout()

# Save in multiple formats
fig.savefig('rainfall_annual.png', dpi=150, bbox_inches='tight')
fig.savefig('rainfall_annual.pdf', bbox_inches='tight')
fig.savefig('rainfall_annual.svg', bbox_inches='tight')

for fmt in ['png', 'pdf', 'svg']:
    fname = f'rainfall_annual.{fmt}'
    if os.path.exists(fname):
        size_kb = os.path.getsize(fname) / 1024
        print(f'Saved: {fname} ({size_kb:.1f} KB)')

plt.show()

### Comparing file sizes at different DPI

In [ ]:
# Compare DPI settings
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(years.year.values, years.values, color='steelblue')
ax.set_title('DPI Comparison Test')
ax.grid(alpha=0.3)

for dpi in [72, 150, 300, 600]:
    fname = f'rainfall_dpi_{dpi}.png'
    fig.savefig(fname, dpi=dpi, bbox_inches='tight')
    size_kb = os.path.getsize(fname) / 1024
    print(f'DPI {dpi:3d}: {fname} ({size_kb:.1f} KB)')
plt.close(fig)

---
## 6. Figure Layout Options

### `figsize` — figure dimensions (width, height) in inches

```python
fig, ax = plt.subplots(figsize=(6, 4))    # standard
fig, ax = plt.subplots(figsize=(12, 4))   # wide (time series)
fig, ax = plt.subplots(figsize=(8, 8))    # square (maps with cartopy)
```

### `constrained_layout` vs `tight_layout` vs `subplots_adjust`

| Method | When to Use |
|--------|-------------|
| `fig.tight_layout()` | Quick auto-adjust of subplot spacing |
| `constrained_layout=True` | Set in `subplots()`, better than tight_layout |
| `fig.subplots_adjust(left, right, top, bottom)` | Manual fine control |
| `plt.subplots_adjust(wspace, hspace)` | Space between subplots |

In [ ]:
# Comparison: different figsize ratios
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

monthly_clim = ts.groupby('time.month').mean()
months = monthly_clim.month.values

sizes = [(4, 4), (6, 4), (10, 4)]
titles = ['Square (4×4)', 'Standard (6×4)', 'Wide (10×4)']

for ax, size, title in zip(axes, sizes, titles):
    ax.plot(months, monthly_clim.values, marker='o', color='darkgreen')
    ax.set_title(f'{title}')
    ax.set_xlabel('Month')
    ax.set_ylabel('mm/month')
    ax.set_xticks(range(1, 13))
    ax.grid(alpha=0.3)

fig.suptitle('Effect of figsize on aspect ratio', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

---
## 7. Exercises

1. **Basic plot** — Plot the full monthly time series for the Addis Ababa point (504 months). Use `figsize=(14, 4)`, a thin line, and a title.
2. **Customise** — Take the plot from Exercise 1 and add:
   - A red dashed horizontal line at the mean
   - Green shaded regions for months above the mean
   - A legend
   - Rotated x-tick labels
3. **Save in 3 formats** — Save the customised figure as PNG (300 dpi), PDF, and SVG.
4. **Subplots** — Create a 1×2 figure with:
   - Left: monthly time series
   - Right: histogram of the same data
5. **Map with customisation** — Plot the climatology map with a different colour map (`'RdYlBu'`) and a colour bar label with units.

In [ ]:
# Exercise 1: Full monthly time series
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts.time.values, ts.values, color='steelblue', linewidth=0.6)
ax.set_title('Monthly Rainfall at (9°N, 38.7°E) — Addis Ababa Region', fontsize=13)
ax.set_xlabel('Time')
ax.set_ylabel('Precipitation (mm/month)')
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Exercise 2: Customised plot
mean_val = float(ts.mean().values)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ts.time.values, ts.values, color='steelblue', linewidth=0.6, label='Monthly rainfall')
ax.axhline(y=mean_val, color='red', linestyle='--', linewidth=1.2, label=f'Mean = {mean_val:.1f} mm')
ax.fill_between(ts.time.values, mean_val, ts.values,
                where=(ts.values > mean_val), color='green', alpha=0.15, label='Above mean')
ax.set_title('Customised Monthly Rainfall Time Series', fontsize=13, fontweight='bold')
ax.set_xlabel('Time')
ax.set_ylabel('Precipitation (mm/month)')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Exercise 3: Save in 3 formats
fig.savefig('rainfall_monthly.png', dpi=300, bbox_inches='tight')
fig.savefig('rainfall_monthly.pdf', bbox_inches='tight')
fig.savefig('rainfall_monthly.svg', bbox_inches='tight')
print('Saved: rainfall_monthly.png (300 dpi), .pdf, .svg')

In [ ]:
# Exercise 4: 1x2 subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(ts.time.values, ts.values, color='steelblue', linewidth=0.6)
ax1.set_title('Monthly Time Series')
ax1.set_xlabel('Time')
ax1.set_ylabel('mm/month')
ax1.grid(alpha=0.3)

ax2.hist(ts.values, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
ax2.set_title('Histogram')
ax2.set_xlabel('mm/month')
ax2.set_ylabel('Count')
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# Exercise 5: Map with RdYlBu colour map
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig, ax = plt.subplots(figsize=(9, 7),
                       subplot_kw={'projection': ccrs.PlateCarree()})

clim = ds.precip.mean(dim='time')
im = ax.pcolormesh(ds.longitude, ds.latitude, clim.values,
                   cmap='RdYlBu', shading='auto')
ax.coastlines(resolution='50m', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.6)
fig.colorbar(im, ax=ax, label='Mean monthly precipitation (mm/month)', shrink=0.7)
ax.set_title('CHIRPS Climatology (1981–2022) — RdYlBu', fontsize=13, fontweight='bold')

fig.tight_layout()
plt.show()

---
## Mini-Project: Publication-Quality Figure

Create a **publication-ready** figure that tells a story about Ethiopian rainfall:

1. **2×2 subplot layout** using `fig, axes = plt.subplots(2, 2, figsize=(12, 10))`
2. Panels:
   - **Top-left:** Climatology map (colour map, coastlines, colour bar, labels)
   - **Top-right:** Time series at a chosen point (with trend line overlaid)
   - **Bottom-left:** Seasonal cycle (monthly means with error bars or std shading)
   - **Bottom-right:** Histogram of annual mean rainfall across all pixels
3. Use consistent fonts, colours, and styling across all panels
4. Add a figure-level title with `fig.suptitle()`
5. Save as `publication_figure.png` at 300 dpi and `publication_figure.pdf`

In [ ]:
# Mini-Project: Publication-quality figure
fig, axes = plt.subplots(2, 2, figsize=(12, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})

clim = ds.precip.mean(dim='time')

# Select a point in the highlands
ts_highland = ds.precip.sel(latitude=11.5, longitude=39.5, method='nearest')
years_highland = ts_highland.groupby('time.year').mean()
monthly_clim = ts_highland.groupby('time.month').mean()
monthly_std = ts_highland.groupby('time.month').std()

# (a) Climatology map
ax0 = axes[0, 0]
im = ax0.pcolormesh(ds.longitude, ds.latitude, clim.values,
                    cmap='viridis', shading='auto')
ax0.coastlines(resolution='50m', linewidth=0.8)
ax0.add_feature(cfeature.BORDERS, linewidth=0.5, alpha=0.6)
cbar = fig.colorbar(im, ax=ax0, label='mm/month', shrink=0.7)
ax0.set_title('(a) Mean Monthly Rainfall', fontsize=12, fontweight='bold')

# (b) Time series with trend
ax1 = fig.add_subplot(2, 2, 2)  # non-projection
ax1.plot(years_highland.year.values, years_highland.values,
         color='steelblue', linewidth=1.5, marker='o', markersize=3, label='Annual mean')
# Linear trend
z = np.polyfit(years_highland.year.values, years_highland.values, 1)
p = np.poly1d(z)
ax1.plot(years_highland.year.values, p(years_highland.year.values),
         color='red', linestyle='--', linewidth=1.5, label=f'Trend: {z[0]:.2f} mm/yr')
ax1.set_title('(b) Annual Rainfall Trend\nHighlands (11.5°N, 39.5°E)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Year')
ax1.set_ylabel('mm/month')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# (c) Seasonal cycle with envelope
ax2 = fig.add_subplot(2, 2, 3)
m = monthly_clim.month.values
ax2.plot(m, monthly_clim.values, color='darkorange', linewidth=2, marker='o', label='Mean')
ax2.fill_between(m,
                 monthly_clim.values - monthly_std.values,
                 monthly_clim.values + monthly_std.values,
                 color='darkorange', alpha=0.2, label='±1 std')
ax2.set_title('(c) Seasonal Cycle', fontsize=12, fontweight='bold')
ax2.set_xlabel('Month')
ax2.set_ylabel('mm/month')
ax2.set_xticks(range(1, 13))
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3)

# (d) Histogram of annual means
annual_mean_map = ds.precip.groupby('time.year').mean(dim='time').mean(dim='year')
ax3 = fig.add_subplot(2, 2, 4)
ax3.hist(annual_mean_map.values.ravel(), bins=80,
         color='steelblue', edgecolor='white', linewidth=0.3)
ax3.set_title('(d) Distribution of Mean Rainfall\nAcross Ethiopia', fontsize=12, fontweight='bold')
ax3.set_xlabel('mm/month')
ax3.set_ylabel('Pixel count')
ax3.grid(alpha=0.3)

fig.suptitle('Ethiopian Rainfall Climatology\nCHIRPS Dataset 1981–2022',
             fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Save the publication figure
fig.savefig('publication_figure.png', dpi=300, bbox_inches='tight')
fig.savefig('publication_figure.pdf', bbox_inches='tight')
print('Saved: publication_figure.png (300 dpi), publication_figure.pdf')

---
## Summary

- The **Object-Oriented interface** (`fig, ax = plt.subplots()`) is the professional way to use matplotlib
- A **Figure** is the container; an **Axes** is a sub-panel within it
- Basic workflow: create fig & ax → plot → customise → display/save
- `savefig()` supports multiple formats (PNG, PDF, SVG) with DPI control
- PNG (100–150 dpi) for web, PNG (300 dpi) or PDF for print
- Real CHIRPS data makes abstract concepts concrete and immediately useful

**Next module:** Line and scatter plots in depth.